# Лабораторная 5 — генеративные модели (полная версия «как надо»)

**Kaggle:** GPU, **Internet ON**.

**NST:** реальные фото (picsum) + репродукции (Wikimedia). Если интернет режется — подключи датасет и укажи `KAGGLE_IMAGES_DIR` в ячейке загрузки.

**GAN:** EMNIST digits, 50 эпох, FID (при OOM уменьши `FID_N`).

In [ ]:
# (Опционально) только если import torch падает — выполни, Restart Session, продолжай со следующей ячейки

import sys, subprocess, os
os.environ["PYTHONNOUSERSITE"] = "1"
for _ in range(2):
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torch", "torchvision", "torchaudio"], capture_output=True)
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--no-cache-dir",
    "torch==2.2.2", "torchvision==0.17.2",
    "--index-url", "https://download.pytorch.org/whl/cu121",
])
print("Restart Session")

In [ ]:
# Окружение

import os
import sys
import urllib.request

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms, models, datasets
from torchvision.utils import make_grid, save_image
from torch.nn.utils import spectral_norm
from PIL import Image

# Полноразмерные картины с Commons > лимита PIL (DecompressionBombError)
Image.MAX_IMAGE_PIXELS = None

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Устройство:", device, "| torch", torch.__version__)

ROOT = "/kaggle/working/lab5_proper" if os.path.isdir("/kaggle/working") else "lab5_proper"
NST_CACHE = os.path.join(ROOT, "nst_images")
os.makedirs(NST_CACHE, exist_ok=True)
os.makedirs(os.path.join(ROOT, "nst"), exist_ok=True)
os.makedirs(os.path.join(ROOT, "gan"), exist_ok=True)
DATA_EMNIST = os.path.join(ROOT, "data_emnist")

In [ ]:
# NST: картинки — свои с Kaggle Input ИЛИ автозагрузка (реальные фото + картины)
# Если загрузил датасет с файлами pair1_content.jpg … — укажи папку:
KAGGLE_IMAGES_DIR = None  # например: "/kaggle/input/nst-lab-images"

EXPECTED = [
    ("pair1_content.jpg", "pair1_style.jpg"),
    ("pair2_content.jpg", "pair2_style.jpg"),
    ("pair3_content.jpg", "pair3_style.jpg"),
]

# Надёжные пары URL: фото (picsum, фиксированные id) + стиль (Wikimedia)
URL_PAIRS = [
    (
        "pair1_content.jpg",
        "https://picsum.photos/id/1018/512/512",
        "pair1_style.jpg",
        "https://upload.wikimedia.org/wikipedia/commons/e/ea/Van_Gogh_-_Starry_Night_-_Google_Art_Project.jpg",
    ),
    (
        "pair2_content.jpg",
        "https://picsum.photos/id/1043/512/512",
        "pair2_style.jpg",
        "https://upload.wikimedia.org/wikipedia/commons/f/f4/The_Scream.jpg",
    ),
    (
        "pair3_content.jpg",
        "https://picsum.photos/id/1036/512/512",
        "pair3_style.jpg",
        [
            "https://upload.wikimedia.org/wikipedia/commons/0/0d/Great_Wave_off_Kanagawa.jpg",
            "https://upload.wikimedia.org/wikipedia/commons/a/a5/Tsunami_by_hokusai_19th_century.jpg",
            "https://upload.wikimedia.org/wikipedia/commons/4/40/The_Kiss_-_Gustav_Klimt_-_Google_Cultural_Institute.jpg",
            "https://picsum.photos/seed/nst-style-wave/512/512",
        ],
    ),
]


def download(url: str, path: str) -> bool:
    try:
        req = urllib.request.Request(
            url,
            headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; rv:91.0) Gecko/20100101 Firefox/91.0"},
        )
        with urllib.request.urlopen(req, timeout=120) as r:
            open(path, "wb").write(r.read())
        return True
    except Exception as e:
        print("Ошибка загрузки:", url[:60], "…", e)
        return False


PAIRS = []
if KAGGLE_IMAGES_DIR and os.path.isdir(KAGGLE_IMAGES_DIR):
    for pc, ps in EXPECTED:
        fc = os.path.join(KAGGLE_IMAGES_DIR, pc)
        fs = os.path.join(KAGGLE_IMAGES_DIR, ps)
        if not (os.path.isfile(fc) and os.path.isfile(fs)):
            raise FileNotFoundError(f"Нет файлов: {fc} или {fs}")
        PAIRS.append((fc, fs))
    print("NST: картинки из Kaggle Input:", KAGGLE_IMAGES_DIR)
else:
    for fn_c, url_c, fn_s, url_s in URL_PAIRS:
        pc = os.path.join(NST_CACHE, fn_c)
        ps = os.path.join(NST_CACHE, fn_s)
        if not os.path.isfile(pc):
            download(url_c, pc)
        if not os.path.isfile(ps):
            urls = url_s if isinstance(url_s, (list, tuple)) else [url_s]
            for u in urls:
                if download(u, ps):
                    break
        if not (os.path.isfile(pc) and os.path.isfile(ps)):
            raise RuntimeError(
                "Не удалось скачать NST-картинки. Включи Internet или создай датасет с "
                + str(EXPECTED)
                + " и задай KAGGLE_IMAGES_DIR."
            )
        PAIRS.append((pc, ps))
    print("NST: скачано в", NST_CACHE)

fig, ax = plt.subplots(3, 2, figsize=(8, 12))
for i, (pc, ps) in enumerate(PAIRS):
    ax[i, 0].imshow(Image.open(pc))
    ax[i, 0].set_title(f"Пара {i+1} content")
    ax[i, 0].axis("off")
    ax[i, 1].imshow(Image.open(ps))
    ax[i, 1].set_title(f"Пара {i+1} style")
    ax[i, 1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# NST: VGG + run_nst (веса слоёв стиля, устойчивость к NaN)

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]
SZ = 256


def load_rgb(path, size=SZ):
    im = Image.open(path).convert("RGB")
    return transforms.Compose(
        [
            transforms.Resize((size, size)),
            transforms.ToTensor(),
            transforms.Normalize(imagenet_mean, imagenet_std),
        ]
    )(im).unsqueeze(0)


def to_rgb01(t):
    m = torch.tensor(imagenet_mean, device=t.device).view(1, 3, 1, 1)
    s = torch.tensor(imagenet_std, device=t.device).view(1, 3, 1, 1)
    x = (t * s + m).clamp(0, 1)
    return x.squeeze(0).permute(1, 2, 0).detach().float().cpu().numpy()


print("Загрузка VGG19…")
vgg = models.vgg19(weights=models.VGG19_Weights.DEFAULT).features.to(device)
for p in vgg.parameters():
    p.requires_grad = False
vgg.eval()

style_layers = {"conv1_1": 0, "conv2_1": 5, "conv3_1": 10, "conv4_1": 19, "conv5_1": 28}
content_layers = {"conv4_2": 21}
all_layers = {**style_layers, **content_layers}
style_w = {"conv1_1": 1.0, "conv2_1": 0.8, "conv3_1": 0.5, "conv4_1": 0.3, "conv5_1": 0.1}


def get_features(x, model, layers):
    idx = {v: k for k, v in layers.items()}
    mx = max(idx.keys())
    out, t = {}, x
    for i, layer in enumerate(model):
        t = layer(t)
        if i in idx:
            out[idx[i]] = t
        if i == mx:
            break
    return out


def gram(f):
    b, c, h, w = f.shape
    Fm = f.view(b, c, h * w)
    return torch.bmm(Fm, Fm.transpose(1, 2)) / (c * h * w)


def run_nst(c, s, steps=300, alpha=1.0, beta=1e5, from_noise=False, log_every=100):
    gen = torch.randn_like(c).requires_grad_(True) if from_noise else c.clone().requires_grad_(True)
    with torch.no_grad():
        cf = get_features(c, vgg, all_layers)
        sf = get_features(s, vgg, all_layers)
    opt = optim.LBFGS([gen], lr=1.0, max_iter=20)
    mean = torch.tensor(imagenet_mean, device=device).view(1, 3, 1, 1)
    std = torch.tensor(imagenet_std, device=device).view(1, 3, 1, 1)

    def closure():
        opt.zero_grad()
        gf = get_features(gen, vgg, all_layers)
        lc = F.mse_loss(gf["conv4_2"], cf["conv4_2"])
        ls = sum(style_w[L] * F.mse_loss(gram(gf[L]), gram(sf[L])) for L in style_layers)
        loss = alpha * lc + beta * ls
        loss.backward()
        closure.last = float(loss.detach().cpu())
        return loss

    for step in range(steps):
        opt.step(closure)
        with torch.no_grad():
            gen.data = (gen.data * std + mean).clamp(0, 1)
            gen.data = (gen.data - mean) / std
            if not torch.isfinite(gen).all():
                gen.data = torch.nan_to_num(gen.data, nan=0.0, posinf=10.0, neginf=-10.0)
                gen.data = (gen.data * std + mean).clamp(0, 1)
                gen.data = (gen.data - mean) / std
        if step % log_every == 0:
            print("  step", step, "loss", closure.last)

    arr = to_rgb01(gen)
    arr = np.nan_to_num(arr, nan=0.5, posinf=1.0, neginf=0.0)
    return np.clip(arr.astype(np.float64), 0.0, 1.0)

In [ ]:
# NST: три пары — content | style | result

fig, ax = plt.subplots(3, 3, figsize=(15, 15))
for i, (pc, ps) in enumerate(PAIRS):
    c = load_rgb(pc).to(device)
    s = load_rgb(ps).to(device)
    ax[i, 0].imshow(np.asarray(Image.open(pc).resize((SZ, SZ))))
    ax[i, 1].imshow(np.asarray(Image.open(ps).resize((SZ, SZ))))
    print(f"NST {i+1}/3…")
    out = run_nst(c, s, beta=1e5, log_every=100)
    ax[i, 2].imshow(out, vmin=0, vmax=1, interpolation="bilinear")
    for j in range(3):
        ax[i, j].axis("off")
ax[0, 0].set_title("content")
ax[0, 1].set_title("style")
ax[0, 2].set_title("result")
plt.tight_layout()
plt.savefig(os.path.join(ROOT, "nst", "three_pairs.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# NST: эволюция по шагам (первая пара)

def run_nst_snaps(c, s, steps=300, beta=1e5, snap_steps=None):
    if snap_steps is None:
        snap_steps = [0, 50, 100, 150, 200, 250, 299]
    gen = c.clone().requires_grad_(True)
    shots = [(0, np.clip(np.nan_to_num(to_rgb01(gen)), 0, 1))]
    with torch.no_grad():
        cf = get_features(c, vgg, all_layers)
        sf = get_features(s, vgg, all_layers)
    opt = optim.LBFGS([gen], lr=1.0, max_iter=20)
    mean = torch.tensor(imagenet_mean, device=device).view(1, 3, 1, 1)
    std = torch.tensor(imagenet_std, device=device).view(1, 3, 1, 1)

    def closure():
        opt.zero_grad()
        gf = get_features(gen, vgg, all_layers)
        lc = F.mse_loss(gf["conv4_2"], cf["conv4_2"])
        ls = sum(style_w[L] * F.mse_loss(gram(gf[L]), gram(sf[L])) for L in style_layers)
        loss = lc + beta * ls
        loss.backward()
        return loss

    for step in range(steps):
        opt.step(closure)
        with torch.no_grad():
            gen.data = (gen.data * std + mean).clamp(0, 1)
            gen.data = (gen.data - mean) / std
            if not torch.isfinite(gen).all():
                gen.data = torch.nan_to_num(gen.data, nan=0.0, posinf=10.0, neginf=-10.0)
                gen.data = (gen.data * std + mean).clamp(0, 1)
                gen.data = (gen.data - mean) / std
        if step in snap_steps and step != 0:
            a = to_rgb01(gen)
            shots.append((step, np.clip(np.nan_to_num(a), 0, 1)))
    return shots


pc, ps = PAIRS[0]
c0 = load_rgb(pc).to(device)
s0 = load_rgb(ps).to(device)
shots = run_nst_snaps(c0, s0)
fig, ax = plt.subplots(1, len(shots), figsize=(3 * len(shots), 3))
for a, (st, rgb) in zip(np.atleast_1d(ax), shots):
    a.imshow(rgb, vmin=0, vmax=1)
    a.set_title(f"step {st}")
    a.axis("off")
plt.suptitle("Эволюция NST")
plt.tight_layout()
plt.savefig(os.path.join(ROOT, "nst", "evolution.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# NST: β и инициализация

c = load_rgb(PAIRS[0][0]).to(device)
s = load_rgb(PAIRS[0][1]).to(device)

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
for j, beta in enumerate([1e3, 1e5, 1e7]):
    print(f"β = {beta}")
    ax[j].imshow(run_nst(c, s, beta=beta, log_every=150), vmin=0, vmax=1)
    ax[j].set_title(f"β={beta:.0e}")
    ax[j].axis("off")
plt.suptitle("Одна пара: влияние β")
plt.tight_layout()
plt.savefig(os.path.join(ROOT, "nst", "beta.png"), dpi=150, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].imshow(run_nst(c, s, from_noise=False, log_every=150), vmin=0, vmax=1)
ax[0].set_title("init = content")
ax[1].imshow(run_nst(c, s, from_noise=True, log_every=150), vmin=0, vmax=1)
ax[1].set_title("init = noise")
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(ROOT, "nst", "init.png"), dpi=150, bbox_inches="tight")
plt.show()

### Выводы по NST (заполни своими наблюдениями)

1. **Три пары:** что сохранилось из content, что пришло из style (текстура, цвет, крупные формы).
2. **β:** при малом β результат ближе к фото; при большом — сильнее «картина», может страдать геометрия.
3. **Инициализация:** от content сходится быстрее и стабильнее; от шума — другая картинка, иногда хуже структура.

## Часть 2 — DCGAN, EMNIST digits

In [ ]:
# EMNIST + DataLoader

tfm = transforms.Compose(
    [
        transforms.Resize(32),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5]),
    ]
)
ds = datasets.EMNIST(
    root=DATA_EMNIST,
    split="digits",
    train=True,
    download=True,
    transform=tfm,
)
loader = DataLoader(
    ds,
    batch_size=128,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
)
print("Размер выборки:", len(ds))
xb, _ = next(iter(loader))
plt.figure(figsize=(8, 8))
plt.axis("off")
plt.imshow(
    np.transpose(
        make_grid(xb[:64], nrow=8, normalize=True, value_range=(-1, 1)).cpu().numpy(),
        (1, 2, 0),
    )
)
plt.show()

In [ ]:
# DCGAN: модели

class Generator(nn.Module):
    def __init__(self, z=100, ch=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(z, 256, 4, 1, 0, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            nn.ConvTranspose2d(64, ch, 4, 2, 1, bias=False),
            nn.Tanh(),
        )

    def forward(self, z):
        return self.net(z)


class Discriminator(nn.Module):
    def __init__(self, ch=1):
        super().__init__()
        self.net = nn.Sequential(
            spectral_norm(nn.Conv2d(ch, 64, 4, 2, 1, bias=False)),
            nn.LeakyReLU(0.2, True),
            spectral_norm(nn.Conv2d(64, 128, 4, 2, 1, bias=False)),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, True),
            spectral_norm(nn.Conv2d(128, 256, 4, 2, 1, bias=False)),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, True),
            spectral_norm(nn.Conv2d(256, 1, 4, 1, 0, bias=False)),
        )

    def forward(self, x):
        return self.net(x).view(-1)


def wi(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(m.weight, 0, 0.02)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.normal_(m.weight, 1, 0.02)
        nn.init.zeros_(m.bias)


z_dim = 100
G = Generator(z_dim, 1).to(device)
D = Discriminator(1).to(device)
G.apply(wi)
D.apply(wi)
crit = nn.BCEWithLogitsLoss()
oG = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
oD = optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
fix_z = torch.randn(16, z_dim, 1, 1, device=device)

In [ ]:
# DCGAN: обучение (50 эпох; для теста поставь EPOCHS = 10)

EPOCHS = 50
hist = {k: [] for k in ("D", "G", "Dx", "DGz")}
snap = {}

for ep in range(EPOCHS):
    sD = sG = sDx = sDGz = 0.0
    n = 0
    for real, _ in loader:
        bs = real.size(0)
        real = real.to(device)
        yr = torch.full((bs,), 0.9, device=device)
        yf = torch.zeros(bs, device=device)

        z = torch.randn(bs, z_dim, 1, 1, device=device)
        fake = G(z)
        dr, df = D(real), D(fake.detach())
        loss_d = crit(dr, yr) + crit(df, yf)
        oD.zero_grad()
        loss_d.backward()
        oD.step()

        z = torch.randn(bs, z_dim, 1, 1, device=device)
        fake = G(z)
        dg = D(fake)
        loss_g = crit(dg, torch.ones(bs, device=device))
        oG.zero_grad()
        loss_g.backward()
        oG.step()

        sD += loss_d.item()
        sG += loss_g.item()
        sDx += torch.sigmoid(dr).mean().item()
        sDGz += torch.sigmoid(dg).mean().item()
        n += 1

    hist["D"].append(sD / n)
    hist["G"].append(sG / n)
    hist["Dx"].append(sDx / n)
    hist["DGz"].append(sDGz / n)
    print(f"{ep+1}/{EPOCHS}  D={hist['D'][-1]:.4f}  G={hist['G'][-1]:.4f}")

    if (ep + 1) in (1, 5, 10, 20, 50):
        with torch.no_grad():
            snap[ep + 1] = make_grid(G(fix_z), nrow=4, normalize=True, value_range=(-1, 1)).cpu()

torch.save({"G": G.state_dict(), "D": D.state_dict(), "hist": hist}, os.path.join(ROOT, "gan", "dcgan.pth"))

In [ ]:
# DCGAN: графики, сетки, 64 изображения, интерполяция, FID

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(hist["D"], label="D")
ax[0].plot(hist["G"], label="G")
ax[0].legend()
ax[0].grid(True, alpha=0.3)
ax[1].plot(hist["Dx"], label="D(x)")
ax[1].plot(hist["DGz"], label="D(G(z))")
ax[1].axhline(0.5, color="k", ls="--", alpha=0.4)
ax[1].legend()
ax[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(ROOT, "gan", "curves.png"), dpi=150)
plt.show()

if snap:
    fig, ax = plt.subplots(1, len(snap), figsize=(4 * len(snap), 4))
    for a, e in zip(np.atleast_1d(ax), sorted(snap.keys())):
        g = snap[e].numpy()
        a.imshow(np.transpose(g, (1, 2, 0)))
        a.set_title(f"ep {e}")
        a.axis("off")
    plt.tight_layout()
    plt.savefig(os.path.join(ROOT, "gan", "epochs.png"), dpi=150, bbox_inches="tight")
    plt.show()

G.eval()
with torch.no_grad():
    big = G(torch.randn(64, z_dim, 1, 1, device=device))
plt.figure(figsize=(9, 9))
plt.axis("off")
plt.imshow(
    np.transpose(
        make_grid(big.cpu(), nrow=8, normalize=True, value_range=(-1, 1)).numpy(),
        (1, 2, 0),
    )
)
plt.savefig(os.path.join(ROOT, "gan", "grid64.png"), dpi=150, bbox_inches="tight")
plt.show()

z1 = torch.randn(1, z_dim, 1, 1, device=device)
z2 = torch.randn(1, z_dim, 1, 1, device=device)
with torch.no_grad():
    inter = G(torch.cat([(1 - a) * z1 + a * z2 for a in torch.linspace(0, 1, 10)], dim=0))
plt.figure(figsize=(14, 2))
plt.axis("off")
plt.imshow(
    np.transpose(
        make_grid(inter.cpu(), nrow=10, normalize=True, value_range=(-1, 1)).numpy(),
        (1, 2, 0),
    )
)
plt.savefig(os.path.join(ROOT, "gan", "interp.png"), dpi=150, bbox_inches="tight")
plt.show()


def denorm_gan(t):
    return (t * 0.5 + 0.5).clamp(0, 1)


try:
    from torchmetrics.image.fid import FrechetInceptionDistance
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torchmetrics"])
    from torchmetrics.image.fid import FrechetInceptionDistance

FID_N = 10_000
fid = FrechetInceptionDistance(feature=2048).to(device)


def prep_fid(x):
    x = denorm_gan(x)
    x = x.repeat(1, 3, 1, 1)
    x = F.interpolate(x, (75, 75), mode="bilinear", align_corners=False)
    return (x * 255).byte()


k = 0
for real, _ in loader:
    fid.update(prep_fid(real.to(device)), real=True)
    k += real.size(0)
    if k >= FID_N:
        break

k = 0
with torch.no_grad():
    while k < FID_N:
        f = G(torch.randn(128, z_dim, 1, 1, device=device))
        fid.update(prep_fid(f), real=False)
        k += f.size(0)

print(f"FID (N={FID_N}):", float(fid.compute()))

### Выводы по DCGAN (заполни)

1. Как ведут себя **loss_D** и **loss_G**; есть ли признаки доминирования D или G.
2. Как меняются **D(x)** и **D(G(z))** — приближение к 0.5 у обоих для фейков часто желательно к концу обучения.
3. Качество цифр: разнообразие, артефакты, mode collapse.
4. **FID** при каком **N** посчитан (в коде 10 000; если не хватило памяти — уменьши `FID_N` и напиши фактическое N).
5. Интерполяция в **z**: плавность, осмысленность промежуточных образов.